## 3-2. 応用例: ベル不等式（CHSH 不等式）の破れのシミュレーション

本節では，QURI Parts を用いてベル不等式の代表例である CHSH 不等式の破れをシミュレーションする。量子もつれが，古典的にはもっともらしく思える「局所性」と「実在性」を同時には満たせないことを，具体的な相関の計算を通して確認する。

### 実在性と局所性

実在性とは，物理系の性質が測定前からあらかじめ定まっているという考え方である。局所性とは，空間的に離れた場所で起きる出来事どうしは互いに即座に影響を及ぼさないという考え方である。古典的な直感ではどちらも自然に受け入れられるが，量子もつれ状態を用いた実験では，この二つを同時に成り立たせることができないことが示唆される。

### ベル状態と CHSH 不等式

2量子ビットのベル状態

$$
|\psi^-\rangle = \frac{1}{\sqrt{2}}\left(|01\rangle - |10\rangle\right)
$$

を考える。この状態では，アリスとボブが同じ基底で測定すると強い反相関が現れる。さらに，測定基底を適切に選ぶと，CHSH 相関

$$
\langle C \rangle = \langle AB \rangle + \langle AB' \rangle + \langle A'B \rangle - \langle A'B' \rangle
$$

の絶対値が，局所実在論から導かれる上限 2 を超え，量子力学では最大で $2\sqrt{2}$ に達する。

[図 3.3 プレースホルダ: CHSH 不等式を破る実験のセットアップ]

以下では，アリスの観測量を $A=Z$, $A'=X$ とし，ボブの観測量を

$$
B=\frac{Z+X}{\sqrt{2}},\qquad B'=\frac{Z-X}{\sqrt{2}}
$$

とする。このとき理論的には，ベル状態に対して $|\langle C \rangle| = 2\sqrt{2}$ が得られる。

In [ ]:
import numpy as np
from sympy import nsimplify

from quri_parts.circuit import LinearMappedUnboundParametricQuantumCircuit
from quri_parts.core.operator import Operator, pauli_label
from quri_parts.core.state import quantum_state
from quri_parts.qulacs.estimator import create_qulacs_vector_estimator

obs1 = Operator({pauli_label("Z0 X1"): 1/np.sqrt(2), pauli_label("Z0 Z1"): 1/np.sqrt(2)})
obs2 = Operator({pauli_label("Z0 Z1"): 1/np.sqrt(2), pauli_label("Z0 X1"): -1/np.sqrt(2)})
obs3 = Operator({pauli_label("X0 X1"): 1/np.sqrt(2), pauli_label("X0 Z1"): 1/np.sqrt(2)})
obs4 = Operator({pauli_label("X0 Z1"): 1/np.sqrt(2), pauli_label("X0 X1"): -1/np.sqrt(2)})

obs1, obs2, obs3, obs4

次に，ベル状態を生成し，ボブ側の測定基底を回転させるためのパラメトリック回路を定義する。

In [ ]:
def chsh_circuit(circuit):
    circuit.add_H_gate(0)
    circuit.add_CNOT_gate(0, 1)
    circuit.add_X_gate(0)
    circuit.add_Z_gate(1)
    theta = circuit.add_parameter("theta")
    circuit.add_ParametricRY_gate(1, {theta: 1.0})
    return circuit

circuit = LinearMappedUnboundParametricQuantumCircuit(2)
chsh_circuit(circuit)
bound_circuit = circuit.bind_parameters([0.0])
bound_circuit

この回路に対して期待値推定を行うと，角度 0 のとき CHSH 相関が $-2\sqrt{2}$ となり，不等式が破れていることが確認できる。

In [ ]:
estimator = create_qulacs_vector_estimator()
state = quantum_state(2, circuit=bound_circuit)

exp1 = estimator(obs1, state).value.real
exp2 = estimator(obs2, state).value.real
exp3 = estimator(obs3, state).value.real
exp4 = estimator(obs4, state).value.real
expectation_value = exp1 + exp2 + exp3 - exp4
expectation_value, nsimplify(expectation_value)

さらに回転角 $\theta$ を変化させていくと，相関の値がどのように変わるかを追うことができる。古典的境界は $\pm 2$，量子的な最大値は $\pm 2\sqrt{2}$ であり，角度に応じてその間を滑らかに変化する様子が確認できる。

In [ ]:
angles = list(np.linspace(0, 2 * np.pi, 51))
expectation_value_list = []

for angle in angles:
    circuit = LinearMappedUnboundParametricQuantumCircuit(2)
    chsh_circuit(circuit)
    bound_circuit = circuit.bind_parameters([angle])
    state = quantum_state(2, circuit=bound_circuit)
    exp1 = estimator(obs1, state).value.real
    exp2 = estimator(obs2, state).value.real
    exp3 = estimator(obs3, state).value.real
    exp4 = estimator(obs4, state).value.real
    expectation_value_list.append(exp1 + exp2 + exp3 - exp4)

expectation_value_list[:5]

[図 3.4 プレースホルダ: CHSH witness の角度依存性]

このシミュレーションは，局所実在論に基づく古典的な上限を量子力学が超えうることを，QURI Parts のオブザーバブル定義，パラメトリック回路，期待値推定を通して確認する良い例になっている。